# Notebook 1: Hugging Face NLP for Workshop
## Translation, Summarization, and Text Generation

This notebook is built for a live workshop format.

- We load one model at a time.
- We explain each parameter before using it.
- We compare model sizes qualitatively (speed, fluency, detail).

All models are from Hugging Face and are <= 8B parameters.

### Learning Goals

1. Run NLP tasks using Hugging Face pipelines.
2. Understand why model size can affect quality and runtime.
3. Control outputs with core generation parameters.

### Session Flow

1. Part A: Translation (FLAN-T5 small -> FLAN-T5 base)
2. Part B: Summarization (DistilBART -> BART-large-CNN)
3. Part C: Text generation (DistilGPT2 -> TinyLlama -> Qwen 7B optional)

### Note

This is a teaching notebook, not a benchmarking notebook.

In [ ]:
%pip install -q -U transformers accelerate sentencepiece torch

In [1]:
import time
import torch
from transformers import pipeline

print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    
else:
    print('Running on CPU')

CUDA available: True
GPU: NVIDIA GeForce RTX 4050 Laptop GPU


## Part A: Translation

We start with a simple NLP task: Arabic -> English translation.

Goal in this part:
- Use the same sentence with two model sizes.
- Observe differences in fluency and latency without formal benchmarking.

### Step A1: Load A Smaller Translation Model

Model: `google/flan-t5-small` (~80M)

Parameters to notice:
- `task='text2text-generation'`: T5 models work as instruction-based text-to-text.
- `torch_dtype=torch.float16` on GPU: can reduce memory usage.

In [ ]:
'google/flan-t5-small'

start = time.time()

if torch.cuda.is_available():
    
    
    translator_small = pipeline(
        'text2text-generation',
        model='google/flan-t5-small',
        device_map='auto',
        torch_dtype=torch.float16
    )
    
else:
    translator_small = pipeline('text2text-generation', model='google/flan-t5-small')

print('Loaded model:', 'google/flan-t5-small')
print(f'Load time: {time.time() - start:.2f} sec')

Device set to use cuda:0


Loaded model: google/flan-t5-small
Load time: 7.19 sec


### Step A1 Inference With The Small Model

Now we run one translation example with `flan-t5-small`.

Highlighted generation settings:
- `max_new_tokens=80`: limits output length.
- `do_sample=False`: deterministic output for teaching consistency.

In [4]:
french_sentence = 'LeIA nous aide à résoudre rapidement des problèmes complexes.'

prompt_small = f'Translate this french sentence to English: {french_sentence}'

small_output = translator_small(
    prompt_small,
    max_new_tokens=80,
    do_sample=False
)[0]['generated_text']

print('Input:', french_sentence)
print('Translation (small):', small_output)

Input: LeIA nous aide à résoudre rapidement des problèmes complexes.
Translation (small): The IA helps to resolve quickly the complex problems.


### Step A2: Load A Larger Translation Model

Model: `google/flan-t5-base` (~250M)

Expected classroom observation:
- Usually more stable wording than the small model.
- Usually slower and heavier on memory.

In [5]:
translation_base_model_id = 'google/flan-t5-base'

start = time.time()
if torch.cuda.is_available():
    translator_base = pipeline(
        'text2text-generation',
        model=translation_base_model_id,
        device_map='auto',
        torch_dtype=torch.float16
    )
else:
    translator_base = pipeline('text2text-generation', model=translation_base_model_id)

print('Loaded model:', translation_base_model_id)
print(f'Load time: {time.time() - start:.2f} sec')

prompt_base = f'Translate this French sentence to English: {french_sentence}'
base_output = translator_base(
    prompt_base,
    max_new_tokens=80,
    do_sample=False
)[0]['generated_text']

print('Translation (base):', base_output)

Device set to use cuda:0


Loaded model: google/flan-t5-base
Load time: 3.65 sec
Translation (base): The IA helps us solve complex problems quickly.


### Translation Reflection

Class discussion prompts:
- Which output sounds more natural?
- Which model felt faster on your machine?
- If you deploy on a low-resource device, which tradeoff is acceptable?

## Part B: Summarization

Now we summarize the same paragraph with two model sizes.

Again, the focus is classroom understanding, not strict evaluation.

In [ ]:
summarizer_small_model_id = 'sshleifer/distilbart-cnn-12-6'



start = time.time()

if torch.cuda.is_available():

    summarizer_small = pipeline(

        'summarization',

        model='sshleifer/distilbart-cnn-12-6',

        device_map='auto',

        torch_dtype=torch.float16

    )

else:

    summarizer_small = pipeline('summarization', model=summarizer_small_model_id)



print('Loaded model:', summarizer_small_model_id)

print(f'Load time: {time.time() - start:.2f} sec')


Device set to use cuda:0


Loaded model: sshleifer/distilbart-cnn-12-6
Load time: 3.62 sec


In [ ]:
article_text = (

    'AI systems are increasingly used in education to personalize learning paths, '

    'generate feedback, and automate routine grading tasks. When implemented carefully, '

    'these systems can help teachers focus more on mentoring and less on repetitive work. '

    'However, institutions must address model bias, student privacy, and transparency in '

    'how automated decisions are made. A balanced rollout combines pilot testing, clear '

    'human oversight, and ongoing evaluation of outcomes for different student groups.'

)



summary_small = summarizer_small(
    article_text,
    max_length=70,
    min_length=25,
    do_sample=False

)[0]['summary_text']



print('Summary (DistilBART):')

print(summary_small)


Summary (DistilBART):
 AI systems are increasingly used in education to personalize learning paths, generate feedback, and automate routine grading tasks . When implemented carefully, these systems can help teachers focus more on mentoring and less on repetitive work . Institutions must address model bias, student privacy, and transparency in how automated decisions are made .


### Step B2: Larger Summarizer

Model: `facebook/bart-large-cnn` (~406M)

Expected classroom observation:
- Often gives more complete and polished summaries.
- Usually slower and heavier than DistilBART.

In [8]:
summarizer_large_model_id = 'facebook/bart-large-cnn'



start = time.time()

if torch.cuda.is_available():

    summarizer_large = pipeline(

        'summarization',

        model=summarizer_large_model_id,

        device_map='auto',

        torch_dtype=torch.float16

    )

else:

    summarizer_large = pipeline('summarization', model=summarizer_large_model_id)



print('Loaded model:', summarizer_large_model_id)

print(f'Load time: {time.time() - start:.2f} sec')


Device set to use cuda:0


Loaded model: facebook/bart-large-cnn
Load time: 3.79 sec


In [9]:
summary_large = summarizer_large(

    article_text,

    max_length=70,

    min_length=25,

    do_sample=False

)[0]['summary_text']



print('Summary (BART-large-CNN):')

print(summary_large)


Summary (BART-large-CNN):
 AI systems are increasingly used in education to personalize learning paths, generate feedback, and automate routine grading tasks. When implemented carefully, these systems can help teachers focus more on mentoring and less on repetitive work. institutions must address model bias, student privacy, and transparency in how automated decisions are made.


## Part C: Text Generation

In this section we move from a very small model to larger chat models.

Focus points for students:
- `max_new_tokens`: controls response length.
- `temperature`: controls randomness.
- `top_p`: controls nucleus sampling diversity.

In [10]:
textgen_small_model_id = 'distilgpt2'



start = time.time()

if torch.cuda.is_available():

    textgen_small = pipeline(

        'text-generation',

        model=textgen_small_model_id,

        device_map='auto',

        torch_dtype=torch.float16

    )

else:

    textgen_small = pipeline('text-generation', model=textgen_small_model_id)



print('Loaded model:', textgen_small_model_id)

print(f'Load time: {time.time() - start:.2f} sec')


Device set to use cuda:0


Loaded model: distilgpt2
Load time: 4.23 sec


In [12]:
prompt_textgen = 'Write a short motivational message for students learning machine learning for the first time.'



distilgpt2_output = textgen_small(

    prompt_textgen,

    max_new_tokens=90,

    do_sample=True,

    temperature=1.8,

    top_p=0.95

)[0]['generated_text']



print('Output (DistilGPT2):')

print(distilgpt2_output)


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Output (DistilGPT2):
Write a short motivational message for students learning machine learning for the first time. Learn how students learn this learning model. Be creative and explore. Use a small space with a goal when looking to design things or create cool ideas if done on your project; learn to create interactive interactive or interactive products with a goal but at your fingertips, when creating interactive ideas you just must focus on making those solutions as they're not made using the most innovative methods we could learn on these computers! Read my course.
"Practical Practical


### Step C2: Move To A 1.1B Chat Model

Model: `TinyLlama/TinyLlama-1.1B-Chat-v1.0`

Expected classroom observation:
- Usually much stronger instruction-following than DistilGPT2.
- Clearly heavier than tiny GPT-style baselines.

In [13]:
textgen_tinyllama_model_id = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'



start = time.time()

if torch.cuda.is_available():

    textgen_tinyllama = pipeline(

        'text-generation',

        model=textgen_tinyllama_model_id,

        device_map='auto',

        torch_dtype=torch.float16

    )

else:

    textgen_tinyllama = pipeline('text-generation', model=textgen_tinyllama_model_id)



print('Loaded model:', textgen_tinyllama_model_id)

print(f'Load time: {time.time() - start:.2f} sec')


Device set to use cuda:0


Loaded model: TinyLlama/TinyLlama-1.1B-Chat-v1.0
Load time: 4.45 sec


In [14]:
tinyllama_prompt = f'Instruction: {prompt_textgen}\nResponse:'



tinyllama_output = textgen_tinyllama(

    tinyllama_prompt,

    max_new_tokens=120,

    do_sample=True,

    temperature=0.7,

    top_p=0.9

)[0]['generated_text']



if tinyllama_output.startswith(tinyllama_prompt):

    tinyllama_output = tinyllama_output[len(tinyllama_prompt):].strip()



print('Output (TinyLlama 1.1B):')

print(tinyllama_output)


Output (TinyLlama 1.1B):
Dear Students,
I am excited to share with you some fundamental concepts of machine learning. Learning machine learning is a journey that requires patience, practice, and persistence. However, it is worth it.

Machine learning is a powerful tool that can help you solve complex problems and make data-driven decisions. In this lesson, we will cover the basics of machine learning, including supervised learning, unsupervised learning, and reinforcement learning.

Supervised learning: This type of machine learning is used to predict outcomes based on given input


### Step C3 (Optional): 7B Instruct Model

Model: `Qwen/Qwen2.5-7B-Instruct` (~7.6B)

Use this section if your machine can handle a 7B model.

Tip: If loading fails, skip this step and continue the workshop.

In [ ]:
qwen_model_id = 'Qwen/Qwen2.5-7B-Instruct'

textgen_qwen = None



try:

    start = time.time()

    if torch.cuda.is_available():

        textgen_qwen = pipeline(

            'text-generation',

            model=qwen_model_id,

            device_map='auto',

            torch_dtype=torch.float16

        )

    else:

        textgen_qwen = pipeline('text-generation', model=qwen_model_id)



    print('Loaded model:', qwen_model_id)

    print(f'Load time: {time.time() - start:.2f} sec')

except Exception as exc:

    print('Qwen load skipped due to resource/tokenizer constraints.')

    print('Details:', exc)


In [ ]:
if textgen_qwen is not None:

    qwen_prompt = f'Instruction: {prompt_textgen}\nResponse:'

    qwen_output = textgen_qwen(

        qwen_prompt,

        max_new_tokens=120,

        do_sample=True,

        temperature=0.7,

        top_p=0.9

    )[0]['generated_text']



    if qwen_output.startswith(qwen_prompt):

        qwen_output = qwen_output[len(qwen_prompt):].strip()



    print('Output (Qwen 7B):')

    print(qwen_output)


## End Of Notebook 1: Suggested Class Activities

1. Change one prompt and repeat across all loaded model sizes.
2. Ask students to write a short note on quality vs runtime tradeoff.
3. Decide one model per task for a low-resource local deployment scenario.